In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
pwd=os.getcwd()

In [4]:
data=pd.read_csv(pwd+"//fake-news/train.csv")
data

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1
...,...,...,...,...,...
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...,0
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...,0
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...,0
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal...",1


In [5]:
data.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [6]:
data.isna().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      20800 non-null  int64 
 1   title   20242 non-null  object
 2   author  18843 non-null  object
 3   text    20761 non-null  object
 4   label   20800 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 812.6+ KB


In [11]:
data=data.dropna()

In [12]:
data["label"].value_counts()

0    10361
1     7924
Name: label, dtype: int64

In [14]:
X=data.drop(columns=["label"])
y=data["label"]

In [15]:
X

,id,title,author,text
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...
...,...,...,...,...
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [16]:
X.shape

(18285, 4)

In [17]:
y.shape

(18285,)

In [18]:
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.models import Sequential


In [19]:
voc_size=5000

# Onehot Representation

In [20]:
messages=X.copy()

In [21]:
messages.reset_index(inplace=True)

In [22]:
messages

,index,id,title,author,text
0,0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...
...,...,...,...,...,...
18280,20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...
18281,20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...
18282,20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...
18283,20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [23]:
import nltk,re
from nltk.corpus import stopwords

In [24]:
from nltk.stem import WordNetLemmatizer
wordnet=WordNetLemmatizer()
corpus=[]

for i in range(0,len(messages)):
    if(i%100==0):
        print(i)
    review=re.sub('[^a-zA-Z]',' ',messages["title"][i])
    review=review.lower()
    review=review.split()
    review=[wordnet.lemmatize(word) for word in review if word not in set(stopwords.words("english"))]
    review=" ".join(review)
    corpus.append(review)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200


In [25]:
corpus

['house dem aide even see comey letter jason chaffetz tweeted',
 'flynn hillary clinton big woman campus breitbart',
 'truth might get fired',
 'civilian killed single u airstrike identified',
 'iranian woman jailed fictional unpublished story woman stoned death adultery',
 'jackie mason hollywood would love trump bombed north korea lack trans bathroom exclusive video breitbart',
 'beno hamon win french socialist party presidential nomination new york time',
 'back channel plan ukraine russia courtesy trump associate new york time',
 'obama organizing action partner soros linked indivisible disrupt trump agenda',
 'bbc comedy sketch real housewife isi cause outrage',
 'russian researcher discover secret nazi military base treasure hunter arctic photo',
 'u official see link trump russia',
 'yes paid government troll social medium blog forum website',
 'major league soccer argentine find home success new york time',
 'well fargo chief abruptly step new york time',
 'anonymous donor pay 

In [26]:
onehot_rep=[one_hot(words,voc_size)for words in corpus]
onehot_rep

[[3797, 2395, 1603, 4635, 1715, 4566, 1096, 4785, 1144, 2479],
 [3048, 4749, 4677, 315, 63, 522, 1426],
 [2011, 3205, 3132, 3237],
 [2589, 3716, 3327, 660, 1284, 4619],
 [17, 63, 3918, 3987, 3152, 1706, 63, 1728, 2327, 4644],
 [367,
  2091,
  3594,
  2966,
  17,
  3455,
  888,
  2267,
  2078,
  1446,
  4695,
  172,
  2531,
  2265,
  1426],
 [4904, 4586, 4533, 3791, 4782, 1076, 3253, 620, 1583, 1467, 2381],
 [67, 4095, 945, 70, 2794, 4245, 3455, 2562, 1583, 1467, 2381],
 [4274, 1617, 4908, 3462, 3688, 1339, 768, 3525, 3455, 3102],
 [3931, 3523, 4664, 459, 1273, 434, 4794, 3881],
 [3263, 2876, 2508, 4383, 1366, 2570, 4543, 3804, 2646, 3586, 321],
 [660, 1187, 1715, 1279, 3455, 2794],
 [977, 3602, 2009, 3276, 2345, 645, 1514, 1526, 3952],
 [1046, 4627, 77, 3815, 4939, 4942, 3500, 1583, 1467, 2381],
 [1479, 2138, 2241, 2689, 25, 1583, 1467, 2381],
 [4449, 2580, 3875, 3733, 944, 960, 3879, 3319, 4249, 3046],
 [2087, 3843, 4749],
 [1464, 2400, 4585, 4425, 3455, 3643, 3211, 1426],
 [2573, 383

# Embedding Representaion

In [27]:
sent_len=20
embedded_docs= pad_sequences(onehot_rep,padding="pre",maxlen=sent_len)
embedded_docs

array([[   0,    0,    0, ..., 4785, 1144, 2479],
       [   0,    0,    0, ...,   63,  522, 1426],
       [   0,    0,    0, ..., 3205, 3132, 3237],
       ...,
       [   0,    0,    0, ..., 1583, 1467, 2381],
       [   0,    0,    0, ..., 3386,  360, 3942],
       [   0,    0,    0, ..., 2793, 3152, 3640]])

In [28]:
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 3797,
       2395, 1603, 4635, 1715, 4566, 1096, 4785, 1144, 2479])

In [29]:
dim=40
model=Sequential()
model.add(Embedding(voc_size,dim,input_length=sent_len))
model.add(Bidirectional(LSTM(100)))
model.add(Dense(1, activation="sigmoid"))
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [30]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 20, 40)            200000    
                                                                 
 bidirectional (Bidirectiona  (None, 200)              112800    
 l)                                                              
                                                                 
 dense (Dense)               (None, 1)                 201       
                                                                 
Total params: 313,001
Trainable params: 313,001
Non-trainable params: 0
_________________________________________________________________


In [31]:
len(embedded_docs),y.shape

(18285, (18285,))

In [32]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [33]:
X_final

array([[   0,    0,    0, ..., 4785, 1144, 2479],
       [   0,    0,    0, ...,   63,  522, 1426],
       [   0,    0,    0, ..., 3205, 3132, 3237],
       ...,
       [   0,    0,    0, ..., 1583, 1467, 2381],
       [   0,    0,    0, ..., 3386,  360, 3942],
       [   0,    0,    0, ..., 2793, 3152, 3640]])

In [34]:
y_final

array([1, 0, 1, ..., 0, 1, 1], dtype=int64)

In [35]:
y

0        1
1        0
2        1
3        1
4        1
        ..
20795    0
20796    0
20797    0
20798    1
20799    1
Name: label, Length: 18285, dtype: int64

In [36]:
X_final.shape,y_final.shape

((18285, 20), (18285,))

In [60]:
#embedded_docs.shape

(18285, 20)

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

# Model Training

In [38]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 [==============================] - 7s 26ms/step - loss: 0.3005 - accuracy: 0.8603 - val_loss: 0.1952 - val_accuracy: 0.9193
Epoch 2/10
192/192 [==============================] - 5s 26ms/step - loss: 0.1299 - accuracy: 0.9504 - val_loss: 0.2046 - val_accuracy: 0.9160
Epoch 3/10
192/192 [==============================] - 8s 44ms/step - loss: 0.0838 - accuracy: 0.9702 - val_loss: 0.2295 - val_accuracy: 0.9127
Epoch 4/10
192/192 [==============================] - 10s 53ms/step - loss: 0.0533 - accuracy: 0.9827 - val_loss: 0.2924 - val_accuracy: 0.9122
Epoch 5/10
192/192 [==============================] - 13s 68ms/step - loss: 0.0329 - accuracy: 0.9898 - val_loss: 0.3586 - val_accuracy: 0.9024
Epoch 6/10
192/192 [==============================] - 9s 49ms/step - loss: 0.0234 - accuracy: 0.9929 - val_loss: 0.4375 - val_accuracy: 0.9034
Epoch 7/10
192/192 [==============================] - 6s 33ms/step - loss: 0.0239 - accuracy: 0.9922 - val_loss: 0.4082 - val_accuracy: 0.90

In [39]:
from sklearn.metrics import confusion_matrix

In [40]:
from keras.models import Sequential 
predict=model.predict(X_test)
predict

189/189 [==============================] - 2s 6ms/step


array([[9.9907202e-01],
       [4.7545377e-04],
       [3.0624822e-05],
       ...,
       [3.0867750e-04],
       [9.9954736e-01],
       [9.9981451e-01]], dtype=float32)

In [41]:
predict_classes=(predict>0.5).astype(int)
predict_classes

array([[1],
       [0],
       [0],
       ...,
       [0],
       [1],
       [1]])

In [42]:
confusion_matrix(y_test,predict_classes)

array([[3121,  298],
       [ 254, 2362]], dtype=int64)

In [43]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,predict_classes)

0.9085335542667772

In [46]:
from sklearn.metrics import classification_report
print(classification_report(y_test,predict_classes))

              precision    recall  f1-score   support

           0       0.92      0.91      0.92      3419
           1       0.89      0.90      0.90      2616

    accuracy                           0.91      6035
   macro avg       0.91      0.91      0.91      6035
weighted avg       0.91      0.91      0.91      6035



## DropOut

In [48]:
from tensorflow.keras.layers import Dropout

dim=40
model1=Sequential()
model1.add(Embedding(voc_size,dim,input_length=sent_len))
model1.add(Bidirectional(LSTM(100)))
model1.add(Dropout(0.2))
model1.add(Dense(1, activation="sigmoid"))
model1.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [49]:
model1.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 20, 40)            200000    
                                                                 
 bidirectional_2 (Bidirectio  (None, 200)              112800    
 nal)                                                            
                                                                 
 dropout (Dropout)           (None, 200)               0         
                                                                 
 dense_1 (Dense)             (None, 1)                 201       
                                                                 
Total params: 313,001
Trainable params: 313,001
Non-trainable params: 0
_________________________________________________________________


In [50]:
model1.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 [==============================] - 9s 34ms/step - loss: 0.3102 - accuracy: 0.8538 - val_loss: 0.1937 - val_accuracy: 0.9181
Epoch 2/10
192/192 [==============================] - 5s 27ms/step - loss: 0.1284 - accuracy: 0.9500 - val_loss: 0.2034 - val_accuracy: 0.9213
Epoch 3/10
192/192 [==============================] - 5s 28ms/step - loss: 0.0849 - accuracy: 0.9708 - val_loss: 0.2471 - val_accuracy: 0.9135
Epoch 4/10
192/192 [==============================] - 6s 32ms/step - loss: 0.0523 - accuracy: 0.9829 - val_loss: 0.2772 - val_accuracy: 0.9145
Epoch 5/10
192/192 [==============================] - 6s 31ms/step - loss: 0.0313 - accuracy: 0.9903 - val_loss: 0.3870 - val_accuracy: 0.9102
Epoch 6/10
192/192 [==============================] - 7s 36ms/step - loss: 0.0229 - accuracy: 0.9931 - val_loss: 0.4311 - val_accuracy: 0.9117
Epoch 7/10
192/192 [==============================] - 6s 34ms/step - loss: 0.0155 - accuracy: 0.9960 - val_loss: 0.4344 - val_accuracy: 0.9102

In [51]:
from keras.models import Sequential 
predict=model1.predict(X_test)
predict

189/189 [==============================] - 2s 7ms/step


array([[9.9971443e-01],
       [1.0076123e-03],
       [2.1591627e-04],
       ...,
       [8.2606514e-04],
       [9.9981403e-01],
       [9.9994165e-01]], dtype=float32)

In [52]:
predict_classes=(predict>0.5).astype(int)
predict_classes

array([[1],
       [0],
       [0],
       ...,
       [0],
       [1],
       [1]])

In [53]:
confusion_matrix(y_test,predict_classes)

array([[3139,  280],
       [ 290, 2326]], dtype=int64)

In [54]:
accuracy_score(y_test,predict_classes)

0.9055509527754764

In [55]:
print(classification_report(y_test,predict_classes))

              precision    recall  f1-score   support

           0       0.92      0.92      0.92      3419
           1       0.89      0.89      0.89      2616

    accuracy                           0.91      6035
   macro avg       0.90      0.90      0.90      6035
weighted avg       0.91      0.91      0.91      6035

